In [ ]:
# -*- coding: utf-8 -*-
# ИНФЕРЕНС ДЛЯ ПОЛУЧЕНИЯ SUBMISSION.CSV

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import librosa
from tqdm import tqdm
from torchvision import models

# ============ НАСТРОЙКИ ============
DEVICE = 'cpu'
DATA_PATH = '/kaggle/input/competitions/birdclef-2026'
MODEL_PATH = '/kaggle/input/your-model/birdclef_best_model.pth'  # Укажите путь к вашей модели

# ============ ЗАГРУЗКА ============
taxonomy = pd.read_csv(os.path.join(DATA_PATH, 'taxonomy.csv'))
species_list = taxonomy['primary_label'].tolist()
print(f"✅ Загружено {len(species_list)} видов")

# ============ МОДЕЛЬ ============
class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes=234):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)
    def forward(self, x):
        return self.backbone(x)

model = BirdCLEFModel(num_classes=len(species_list))
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()
print("✅ Модель загружена")

# ============ ФУНКЦИИ ============
def load_audio(file_path, target_sr=32000, duration=5.0):
    try:
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)
        target_length = int(target_sr * duration)
        if waveform.shape[1] < target_length:
            padding = target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :target_length]
        return waveform.squeeze().numpy().astype(np.float32), target_sr
    except:
        return np.zeros(int(target_sr * duration), dtype=np.float32), target_sr

def create_mel_spectrogram(audio, sr=32000, n_mels=128):
    hop_length = 512
    win_length = 2048
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        hop_length=hop_length, win_length=win_length
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-8)
    return mel_spec_db.astype(np.float32)

# ============ ИНФЕРЕНС ============
test_dir = os.path.join(DATA_PATH, 'test_soundscapes')
test_files = [f for f in os.listdir(test_dir) if f.endswith('.ogg')]
print(f"📁 Найдено тестовых файлов: {len(test_files)}")

all_preds = []
all_row_ids = []

for file_name in tqdm(test_files, desc="Обработка"):
    file_path = os.path.join(test_dir, file_name)
    audio, sr = load_audio(file_path, duration=60.0)
    window_samples = int(sr * 5.0)

    for start in range(0, len(audio) - window_samples + 1, window_samples):
        window = audio[start:start + window_samples]
        mel_spec = create_mel_spectrogram(window, sr)
        input_tensor = torch.FloatTensor(mel_spec).unsqueeze(0).unsqueeze(0)
        input_tensor = nn.functional.interpolate(input_tensor, size=(224, 224), mode='bilinear')

        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()[0]

        all_preds.append(probs)
        row_id = f"{file_name[:-4]}_{int((start + window_samples) / sr)}"
        all_row_ids.append(row_id)

submission_df = pd.DataFrame(all_preds, columns=species_list)
submission_df.insert(0, 'row_id', all_row_ids)
submission_df.to_csv('submission.csv', index=False)
print(f"\n✅ submission.csv создан! Строк: {len(submission_df)}")
